In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) ->str:
    """Read an email from the given address."""
    
    return runtime.state["email"]

@tool
def send_email(body: str) ->str:
    """Send an email to the given address with the given subject and body."""

    return f"Email sent"


In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class Email_State(AgentState):
    email:str

agent= create_agent(
    model= "gpt-5-nano",
    checkpointer= InMemorySaver(),
    state_schema= Email_State,
    tools= [read_email, send_email],
    middleware= [
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ]
)

In [16]:
from langchain.messages import HumanMessage

config={"configurable": {"thread_id": "5"}}

response= agent.invoke(
    {
        "messages": [
            HumanMessage(content= "Please read my email and send a response immediately. Send the reply now in the same thread.")
        ],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    
        config=config
    
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Re: '
                                                                          'Meeting '
                                                                          'reschedule\n'
                                                                          '\n'
                                                                          'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
          

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Re: Meeting reschedule\n\nHi John,\n\nThanks for letting me know. I can reschedule. What time works best for you tomorrow? If you'd like, I can propose a couple of options:\n\n- 10:00 AM\n- 2:00 PM\n\nPlease let me know which option you prefer, or suggest another time that suits you. I’ll adjust the calendar invite accordingly.\n\nBest,\nSeán"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Re: Meeting reschedule\\n\\nHi John,\\n\\nThanks for letting me know. I can reschedule. What time works best for you tomorrow? If you\'d like, I can propose a couple of options:\\n\\n- 10:00 AM\\n- 2:00 PM\\n\\nPlease let me know which option you prefer, or suggest another time that suits you. I’ll adjust the calendar invite accordingly.\\n\\nBest,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='66

In [7]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Re: Meeting reschedule

Hi John,

Thanks for letting me know. I can reschedule. What time works best for you tomorrow? If you'd like, I can propose a couple of options:

- 10:00 AM
- 2:00 PM

Please let me know which option you prefer, or suggest another time that suits you. I’ll adjust the calendar invite accordingly.

Best,
Seán


## Approve

In [8]:
from langgraph.types import Command

response= agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "approve"}]
        }
    ),
    config= config
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='3081fc7a-7890-437e-a318-99d412f978e3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 167, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrBScnj8ov1yw6CoEb1qGrCGH79bw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ecdb7-6910-7671-a2c4-0cd92bc405cc-0', tool_calls

## Reject

In [12]:
response= agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject",
                           "message": "No please sign off - Your merciful leader, Seán."}]
        }
    ),
    config= config
)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
      

## Edit

In [17]:
response= agent.invoke(
    Command(
        resume=
        {
            "decisions": [{
                "type": "edit",
                "edited_action":{
                    "name": "send_email",
                    "args": {"body": "This is the last straw, you're fired!"}
                }
            }]
        }
    ),
    config= config
)
pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='d96bfb29-9df7-4745-ad51-d6bd9564c1f8'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 787, 'prompt_tokens': 167, 'total_tokens': 954, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrBuD5FA1tiQ6DS2ScKQ8MivWEir9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ecdd1-866a-7d02-9af6-965b088a12af-0', tool_calls